### Dataset

In [1]:
from torchvision import datasets, transforms

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, 
                                  download=True, transform=transform)
test_dataset  = datasets.CIFAR10(root='./data', train=False, 
                                  download=True, transform=transform)

100.0%
c:\Users\Lucio\miniconda3\envs\VisionArtificialAvanzada\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [ ]:
import numpy as np
from torch.utils.data import DataLoader

TASKS = [
    [0, 1],   # airplane, automobile
    [2, 3],   # bird, cat
    [4, 5],   # deer, dog
    [6, 7],   # frog, horse
    [8, 9],   # ship, truck
]

def get_task_data(dataset, task_classes):
    targets = np.array(dataset.targets)
    mask = np.isin(targets, task_classes)
    indices = np.where(mask)[0]
    return torch.utils.data.Subset(dataset, indices)

def get_data_loaders(subset_train, subset_test, val_size=0.1, batch_size=64, seed=42):
    np.random.seed(seed)
    num_samples = len(subset_train)
    indices = list(range(num_samples))
    np.random.shuffle(indices)
    
    split = int(np.floor(val_size * num_samples))
    train_indices, val_indices = indices[split:], indices[:split]
    
    train_subset = torch.utils.data.Subset(subset_train, train_indices)
    val_subset = torch.utils.data.Subset(subset_train, val_indices)
    
    train_loader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False)
    test_loader = torch.utils.data.DataLoader(subset_test, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader

In [ ]:
import torch
dataloaders = []
for task_classes in TASKS:
    train_subset = get_task_data(train_dataset, task_classes)
    test_subset  = get_task_data(test_dataset, task_classes)
    
    train_loader, val_loader, test_loader = get_data_loaders(train_subset, test_subset)
    
    dataloaders.append((train_loader, val_loader, test_loader))

    print(f"Tarea {task_classes}: {len(train_subset)} muestras de entrenamiento, {len(test_subset)} muestras de prueba")

Tarea [0, 1]: 10000 muestras de entrenamiento, 2000 muestras de prueba
Tarea [2, 3]: 10000 muestras de entrenamiento, 2000 muestras de prueba
Tarea [4, 5]: 10000 muestras de entrenamiento, 2000 muestras de prueba
Tarea [6, 7]: 10000 muestras de entrenamiento, 2000 muestras de prueba
Tarea [8, 9]: 10000 muestras de entrenamiento, 2000 muestras de prueba
